## 1. Search an Entity Graph on Azure

An Entity Graph lets you traverse the entities and relationships extracted from the source text, using the types and predicates defined in the ontology. This notebook asks "which facility holds which designation or registration, and where is it located?" in two ways, with hand-written GQL (Graph Query Language) and with natural-language Agentic Retrieval, and traces the results back to the evidence chunks.

```mermaid
flowchart TD
    D[Hand-written GQL] --> G[Traverse the managed Entity Graph]
    Q[Natural-language question] --> KB[Knowledge base retrieve]
    KB --> P[The model converts the question to GQL]
    P --> G
    G --> R[Graph rows and evidence chunks]
    P --> A[Check the generated GQL and the execution history in activity]
```

| What to check | Direct GQL (chapter 3) | Agentic Retrieval (chapter 4) |
|---|---|---|
| Input | Hand-written GQL that specifies types and predicates | A natural-language question placed in `messages` |
| Search target | The graph index generated by the knowledge source | A KB that references an `azureBlob` source with graph extraction enabled |
| Output to observe | Graph rows, diagnostics, and the evidence chunks reachable from each row | The generated GQL in `activity`, and the graph rows and evidence chunks in `references` |

If direct GQL finds a path but natural-language retrieval does not, compare the generated GQL in `activity` with your hand-written GQL. This shows where the type, predicate or filter condition differs.


This notebook uses the same **`azure-search-documents==12.2.0b1`** as Functional Test 1. It reads definitions with `SearchIndexClient`, runs direct GQL with `SearchClient.graph()`, and runs Agentic Retrieval with `KnowledgeBaseRetrievalClient.retrieve()`. You can inspect requests and responses with the `.as_dict()` method of the SDK models. The Copilot plugin is not required.

Source: [Entity Graph](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph) / [Retrieval reasoning effort](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)

## 2. Connection settings and the boundary of changes

### Azure prerequisites

| Item | What you need |
|---|---|
| Service | A Foundry IQ / Azure AI Search service that supports `2026-11-01-preview` and the graph features |
| Entity Graph | A graph-enabled source that was already synced in Functional Test 1. The default name is `ft-entity-ks`. If your source has a different name, set `FOUNDRY_IQ_ENTITY_SOURCE`. If you use a different ontology, adapt the GQL in chapter 3 and the question in chapter 4 to that definition |
| Retrieval model (chapter 4 only) | An Azure OpenAI chat deployment. You specify it on the knowledge base side, separately from the model used for graph extraction |
| Running user | You authenticate with Microsoft Entra ID. Check that you have Search Index Data Reader for retrieval, and the corresponding Search Service Contributor permissions for reading source and index definitions and creating the KB |
| Search managed identity (chapter 4 only) | To connect to the model, it needs the Cognitive Services OpenAI User role on Azure OpenAI |

This notebook does not create services or models, assign roles, upload documents, re-extract the graph, or delete resources. For the graph setup, see [Functional Test 1](./functional-test-1-entity-graph.ipynb).

### Kernel dependencies

If a live cell raises an import error, or if the SDK is not `12.2.0b1`, run the following in the notebook kernel you selected.

```python
%pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv
```

Use [.env.example](./.env.example) as a guide and set the required entries in the `.env` file in the same folder as the notebook. Chapter 3 needs `AZURE_SEARCH_ENDPOINT`. Chapter 4 also needs `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT` and `AZURE_OPENAI_MODEL`. Prepare authentication for the target tenant that `DefaultAzureCredential` uses, and never write credentials or access tokens in a cell. Values in `.env` take precedence over environment variables.

### Run switches

- `RUN_AZURE` (this section): With the default `True`, the notebook connects to Azure, reads the existing graph and searches it. Set it to `False` if you do not want to connect to Azure. Searches on Azure and the model calls in chapter 4 may incur charges.
- `CONFIRM_WRITES` (the creation cell in 4.1): After you review the plan, this allows only the new creation of the KB for this notebook. It never overwrites an existing source or KB.

If you start with a new kernel, run the code cells in this section from top to bottom. To try only the natural-language retrieval in chapter 4, you can skip the direct GQL in chapter 3.

Source: [Entity Graph prerequisites and limitations](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [Knowledge base retrieval permissions](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [Search RBAC](https://learn.microsoft.com/azure/search/search-security-rbac) / [Azure Identity authentication in a local development environment](https://learn.microsoft.com/azure/developer/python/sdk/authentication/local-development-environment)

In [ ]:
from __future__ import annotations

import html
import json
import os
import re
from pathlib import Path
from urllib.parse import urlsplit

from IPython.display import HTML, SVG, display


NOTEBOOK_DIR = Path(".")


def show_json(value: object) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def preview_text(value: object) -> object:
    return value[:200] + "…" if isinstance(value, str) and len(value) > 200 else value


def table(rows: list[dict]) -> None:
    if not rows:
        print("There are 0 items to display.")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = "".join(f"<th>{html.escape(str(key))}</th>" for key in columns)
    body = []
    for row in rows:
        cells = []
        for key in columns:
            value = row.get(key, "")
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
            cells.append(f"<td>{html.escape(text)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

In [ ]:
RUN_AZURE = True
API_VERSION = "2026-11-01-preview"

if RUN_AZURE:
    from importlib.metadata import version
    from azure.core import MatchConditions
    from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
    from azure.core.pipeline import PipelineResponse
    from azure.identity import DefaultAzureCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.indexes.models import (
        AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
        KnowledgeSourceReference,
    )
    from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
    from azure.search.documents.knowledgebases.models import (
        AgentRuntimeGraphHydrateDocumentsParams, AgentRuntimeGraphParams,
        AgentRuntimeGraphProvenanceParams, AzureBlobKnowledgeSourceParams,
        KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
        KnowledgeRetrievalLowReasoningEffort,
    )
    from azure.search.documents.models import (
        GraphQueryProvenanceHydrateDocumentsParameters, GraphQueryProvenanceParameters,
    )
    from dotenv import load_dotenv
    if version("azure-search-documents") != "12.2.0b1":
        raise RuntimeError("Install the same azure-search-documents==12.2.0b1 as Functional Test 1 in the kernel.")
    load_dotenv(NOTEBOOK_DIR / ".env", override=True)

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
MODEL_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "")
CHAT_MODEL = os.getenv("AZURE_OPENAI_MODEL", "")
ENTITY_SOURCE = os.getenv("FOUNDRY_IQ_ENTITY_SOURCE", "ft-entity-ks")
ENTITY_KB = os.getenv("FOUNDRY_IQ_ENTITY_KB", "ft-entity-agentic-kb")

if RUN_AZURE:
    endpoint = urlsplit(SEARCH_ENDPOINT)
    if (endpoint.scheme != "https" or not endpoint.hostname or endpoint.username
            or endpoint.password or endpoint.path or endpoint.query or endpoint.fragment or "<" in SEARCH_ENDPOINT):
        raise ValueError("Set AZURE_SEARCH_ENDPOINT to the actual HTTPS service URL.")
    credential = DefaultAzureCredential()
    client_options = {"api_version": API_VERSION, "retry_total": 0, "connection_timeout": 10, "read_timeout": 90}
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, **client_options)
    azure_clients: list[SearchIndexClient | SearchClient | KnowledgeBaseRetrievalClient] = [index_client]

table([{"Azure run": RUN_AZURE, "API version": API_VERSION,
        "Entity source": ENTITY_SOURCE, "KB for Entity Graph": ENTITY_KB}])

### 2.1 SDK exception handling and safe creation of new resources

Authentication and communication are left to the SDK. A service error stops execution with the SDK's `HttpResponseError`, and only a missing resource is handled as `ResourceNotFoundError`. A partial `206` response from `retrieve` is detected by a response hook, which stops execution. Because each client sets `retry_total = 0`, no automatic retries happen.

New KBs are created with `MatchConditions.IfMissing`, so a KB with the same name that was created earlier is never overwritten. If one already exists, the notebook reuses it after matching the settings that the plan in this notebook includes. Default values that the server adds are not compared.

Source: [Search concurrency control](https://learn.microsoft.com/azure/search/search-howto-concurrency) / [Knowledge Retrieval REST](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview)

In [ ]:
def reject_partial_response(response: PipelineResponse) -> None:
    if response.http_response.status_code == 206:
        raise HttpResponseError("Agentic Retrieval returned a partial response. Check the response body and the execution history.",
                                response=response.http_response)


def require_expected(actual: object, expected: object, path: str = "resource") -> None:
    if isinstance(expected, dict) and isinstance(actual, dict):
        for key, value in expected.items():
            require_expected(actual.get(key), value, f"{path}.{key}")
    elif isinstance(expected, list) and isinstance(actual, list) and len(actual) == len(expected):
        for index, value in enumerate(expected):
            require_expected(actual[index], value, f"{path}[{index}]")
    elif actual != expected:
        raise ValueError(f"The existing configuration differs from the plan: {path}. Do not overwrite it. Use a different name or review the configuration.")


def create_or_reuse_kb(definition: KnowledgeBase, *, confirm_writes: bool) -> KnowledgeBase:
    try:
        existing = index_client.get_knowledge_base(definition.name)
    except ResourceNotFoundError:
        print("Not created:", definition.name)
        if not confirm_writes:
            raise RuntimeError("Review the plan in 4.1, set CONFIRM_WRITES to True in the creation cell, and run it again.") from None
        return index_client.create_or_update_knowledge_base(definition, match_condition=MatchConditions.IfMissing)
    require_expected(existing.as_dict(), definition.as_dict())
    print("Matched the configuration and reusing:", definition.name)
    return existing

## 3. Read the Entity Graph directly

### 3.1 Get the actual graph name and index name from the registered source

Do not guess the index name of the managed graph. Read `createdResources.graphIndex` instead. `graphExtraction.graphName` is the name that distinguishes a graph inside the index. The notebook also displays the sync status, but the status alone does not tell you whether the extracted content is correct.

The GQL in this section matches the built-in ontology of Functional Test 1. If the relationships have not been extracted, check the sync, the ontology and the source text in the original notebook.

Source: [Entity Graph created resources and sync verification](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#verify-extraction-completes)

In [ ]:
if RUN_AZURE:
    entity_source = index_client.get_knowledge_source(ENTITY_SOURCE).as_dict()
    if entity_source["kind"] != "azureBlob":
        raise ValueError("Specify an azureBlob source for the Entity Graph.")
    blob_parameters = entity_source["azureBlobParameters"]
    GRAPH_INDEX = blob_parameters["createdResources"]["graphIndex"]
    GRAPH_NAME = blob_parameters["graphExtraction"]["graphName"]
    if not GRAPH_INDEX or not GRAPH_NAME:
        raise ValueError("Check the graph extraction settings and createdResources.")
    table([{"Source": ENTITY_SOURCE, "Graph name": GRAPH_NAME, "Graph index": GRAPH_INDEX}])
    show_json(index_client.get_knowledge_source_status(ENTITY_SOURCE).as_dict())
else:
    print("Not run: reading the registered Entity Graph and its sync status.")

### 3.2 Build the filter from the graph definition

Confirm that `graphProfile.kind = edgeList`, and also get the graph name field from the definition. An OData filter limits the target even when the same index contains other graphs.

Source: [Documents - Graph filter](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [Official SDK filter sample](https://github.com/Azure/azure-sdk-for-python/blob/e1c0306c77b27b46c79755f53783474a03fd8c0d/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_AZURE:
    graph_schema = index_client.get_index(GRAPH_INDEX).as_dict()
    profile = graph_schema["graphProfile"]
    if profile["kind"] != "edgeList":
        raise ValueError("Direct GQL requires a graphProfile of kind edgeList.")
    graph_field = profile["fields"]["graphNameField"]
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", graph_field):
        raise ValueError("Check the identifier of the graph name field.")
    escaped_name = GRAPH_NAME.replace("'", "''")
    GRAPH_FILTER = f"{graph_field} eq '{escaped_name}'"
    graph_client = SearchClient(SEARCH_ENDPOINT, GRAPH_INDEX, credential, **client_options)
    azure_clients.append(graph_client)
    show_json(profile)
    print("Target filter:", GRAPH_FILTER)
else:
    print("Not run: checking the graph definition and the filter.")

### 3.3 Traverse two hops through the facility with GQL

This step retrieves the path `designation <- registered facility -> location`. `registeredAs` and `locatedIn` are predicates of the built-in ontology of this repository. The arrow directions follow the relationship directions of the ontology. The facility node has no type label, and only the directions of the two predicates constrain it.

Direct GQL does not use the retrieval model for natural-language conversion. By confirming the stored relationships first, you can tell apart two causes when the later natural-language retrieval returns nothing: either "the relationship does not exist" or "the generated GQL does not fit".

`LIMIT 10` is the upper limit of graph rows to return. `maxEvidencePerRow` is the upper limit of evidence collection for each row, and it is a **different value**. The `hydrateDocuments` of direct GQL does not specify `top`.

After the search, the notebook draws the paths as SVG from the returned rows. It does not search Azure again for drawing.

Source: [Direct GQL and evidence retrieval](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [REST Graph API](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
DIRECT_GQL = (
    "MATCH (designation:Designation)<-[:registeredAs]-(site)-[:locatedIn]->(place:Place) "
    "RETURN designation.name AS designation, site.name AS site, place.name AS place LIMIT 10"
)
print(DIRECT_GQL)
if RUN_AZURE:
    direct_result = graph_client.graph(
        gql=DIRECT_GQL, filter=GRAPH_FILTER, include_statistics=True, include_diagnostics=True,
        provenance=GraphQueryProvenanceParameters(
            kind="hydrate", max_evidence_per_row=6,
            hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(
                select="uid,snippet,snippet_parent_id,blob_url"),
        ),
    ).as_dict()
    table(direct_result["value"])
    show_json({"Diagnostics": direct_result.get("@search.graphDiagnostics"),
               "Statistics": direct_result.get("@search.graphStats")})
    if any(item.get("severity") == "error" for item in direct_result.get("@search.graphDiagnostics", [])):
        raise RuntimeError("The graph diagnostics contain errors. Check the details above.")
    if not direct_result["value"]:
        print("There are 0 paths. Check the extraction status, the types and the predicates. This is not proof that extraction succeeded.")
else:
    print("Not run: direct GQL on Azure. Only the GQL that would be requested is shown above.")

In [ ]:
def draw_paths(paths: list[list[str]], predicates: list[str], directions: list[int]) -> None:
    if len(predicates) != len(directions) or any(direction not in (-1, 1) for direction in directions):
        raise ValueError("Specify the arrow direction of each relationship as 1 or -1.")
    if any(len(path) != len(predicates) + 1 for path in paths):
        raise ValueError("The number of nodes in a path must be the number of relationships + 1.")
    if not paths:
        print("There are 0 paths to draw.")
        return
    width, height = 300 * (len(predicates) + 1), 110 * len(paths)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="Graph paths found by the search">',
             '<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#345"/></marker></defs>']
    for row_number, path in enumerate(paths):
        y = row_number * 110 + 60
        for column, name in enumerate(path):
            x = column * 300 + 15
            parts.append(f'<rect x="{x}" y="{y - 24}" width="190" height="48" rx="8" fill="#e8f2ff" stroke="#3976b8"/>')
            parts.append(f'<text x="{x + 95}" y="{y + 5}" text-anchor="middle" font-family="sans-serif" font-size="14">{html.escape(str(name))}</text>')
        for column, predicate in enumerate(predicates):
            start, end = column * 300 + 207, (column + 1) * 300 + 12
            if directions[column] == -1:
                start, end = end, start
            parts.append(f'<line x1="{start}" y1="{y}" x2="{end}" y2="{y}" stroke="#345" stroke-width="2" marker-end="url(#arrow)"/>')
            parts.append(f'<text x="{column * 300 + 260}" y="{y - 10}" text-anchor="middle" font-family="sans-serif" font-size="12">{html.escape(predicate)}</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if RUN_AZURE:
    draw_paths([[row["designation"], row["site"], row["place"]] for row in direct_result["value"]],
               ["registeredAs", "locatedIn"], [-1, 1])
else:
    print("Not run: drawing the paths returned by Azure.")

### 3.4 Go from a graph row back to the evidence chunks

The `@search.graphProvenance.evidence` of each row holds the IDs of the edge, the chunk and the parent document. `@search.graphProvenanceDocuments` is a dictionary of source text chunks shared by multiple rows. Read it with `chunkId` as the key.

The table shows the first 200 characters of the source text and truncates the rest with "…". The full text remains in `evidence_rows` and `direct_result`.

`evidenceCount.relation = exact` means that the count is exact. `lowerBound` means that only a lower bound is known, for example because of a collection limit. If the evidence text could not be retrieved, the notebook reports that fact.

Source: [Entity Graph evidence IDs](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [GraphQueryEvidenceCountRelation](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview#graphqueryevidencecountrelation)

In [ ]:
if RUN_AZURE:
    hydrated = direct_result.get("@search.graphProvenanceDocuments", {})
    evidence_rows = []
    for row_number, row in enumerate(direct_result["value"], start=1):
        provenance = row.get("@search.graphProvenance", {})
        print("Row", row_number, "evidence count:", provenance.get("evidenceCount", "Not retrieved"))
        for evidence in provenance.get("evidence", []):
            chunk = hydrated.get(evidence["chunkId"], {})
            evidence_rows.append({"Row": row_number, **evidence,
                                  "Source text": chunk.get("snippet", "Not retrieved"), "Source URL": chunk.get("blob_url", "Not retrieved")})
    table([{**row, "Source text": preview_text(row["Source text"])} for row in evidence_rows])
    if direct_result["value"] and not evidence_rows:
        print("Rows were returned, but no evidence IDs were retrieved. Check the provenance settings and the response.")
else:
    print("Not run: checking the GQL evidence and source text.")

## 4. Use the Entity Graph from Agentic Retrieval

### 4.1 Plan the knowledge base for graph retrieval

**In the current preview, you cannot enable Entity Graph retrieval on an existing knowledge base afterward.** The first time, you create a dedicated `ft-entity-agentic-kb`. Check the source and the model in the plan displayed beforehand, then set `CONFIRM_WRITES = True` in the next creation cell. If a graph KB with the same name already exists, it is reused only when it matches the plan. To use a different name, set `FOUNDRY_IQ_ENTITY_KB` in `.env`.

Configure a retrieval model separately from the chat model that extracted the graph. Even if you reuse the same deployment, the knowledge base needs its own setting. Set `AZURE_OPENAI_MODEL` to the actual model name, which is not necessarily the same as the deployment name. The official documentation recommends `gpt-5.6-luna` with the `low` setting, but this notebook uses your own supported deployment.

With `outputMode = extractiveData`, you first observe the retrieved data and its sources. The notebook adds no step in which a model generates a final natural-language answer.

Source: [Entity Graph limitations](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#limitations) / [Creating a knowledge base](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base)

In [ ]:
if RUN_AZURE:
    entity_kb_plan = KnowledgeBase(
        name=ENTITY_KB, description="agentic-retrieval-graphs notebook: entity graph retrieval",
        knowledge_sources=[KnowledgeSourceReference(name=ENTITY_SOURCE)],
        models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=MODEL_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL))],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(), output_mode="extractiveData",
        retrieval_instructions="Use graph_search for entity relationship questions. Traverse the graph and return evidence for the relationships.",
    )
    show_json(entity_kb_plan.as_dict())
    if not MODEL_ENDPOINT or not CHAT_DEPLOYMENT or not CHAT_MODEL:
        print("The model settings are incomplete. Before the live run, set the actual endpoint, deployment name and model name in .env.")
else:
    print("Not run: planning the KB for the Entity Graph with SDK models.")

In [ ]:
CONFIRM_WRITES = False

if RUN_AZURE:
    if not all((MODEL_ENDPOINT, CHAT_DEPLOYMENT, CHAT_MODEL)) or any("<" in value for value in (MODEL_ENDPOINT, CHAT_DEPLOYMENT, CHAT_MODEL)):
        raise ValueError("The actual Azure OpenAI endpoint, chat deployment and model name are required.")
    entity_kb_definition = create_or_reuse_kb(entity_kb_plan, confirm_writes=CONFIRM_WRITES)
    require_expected(entity_kb_definition.as_dict(), entity_kb_plan.as_dict())
    entity_kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=ENTITY_KB, **client_options)
    azure_clients.append(entity_kb_client)
    print("Verified the configuration of the knowledge base for the graph:", ENTITY_KB)
else:
    print("Not run: creating or matching the knowledge base for the graph.")

### 4.2 Build the natural-language request JSON

This time you ask about the same facility, designation and location relationships as in chapter 3, in natural language. `includeQuery = true` is set to observe the generated GQL, and `includeActivity = true` is set for the execution history.

`hydrateDocuments.top = 10` of the knowledge base is the upper limit on the number of evidence documents to retrieve, not the upper limit on the number of graph rows of the GQL. If you rebuild the request with `hydrate = False`, you can skip the source text retrieval, but searching again incurs additional model calls.

Source: [Retrieve request that uses a graph](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base) / [Official request and response example](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
ENTITY_QUESTION = "For each location, which facilities hold which designations or registrations? Show the facility name, the designation name and the location, together with the evidence for each relationship."


def entity_request(question: str, hydrate: bool = True) -> KnowledgeBaseRetrievalRequest:
    if not question.strip():
        raise ValueError("Enter a question.")
    graph_parameters = AgentRuntimeGraphParams(include_query=True)
    if hydrate:
        graph_parameters.provenance = AgentRuntimeGraphProvenanceParams(
            kind="hydrate", hydrate_documents=AgentRuntimeGraphHydrateDocumentsParams(
                select="uid,snippet,snippet_parent_id,blob_url", top=10),
        )
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
        include_activity=True, output_mode="extractiveData", max_runtime_in_seconds=60,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        knowledge_source_params=[AzureBlobKnowledgeSourceParams(
            knowledge_source_name=ENTITY_SOURCE, include_references=True, include_reference_source_data=True,
            fail_on_error=True, graph_parameters=graph_parameters)],
    )


if RUN_AZURE:
    entity_payload = entity_request(ENTITY_QUESTION)
    show_json(entity_payload.as_dict())
else:
    print("Not run: building the Entity Graph retrieval request with SDK models.")

In [ ]:
if RUN_AZURE:
    entity_result = entity_kb_client.retrieve(entity_payload, raw_response_hook=reject_partial_response).as_dict()
    print("Top-level fields of the response:", list(entity_result))
    show_json(entity_result["response"])
else:
    print("Not run: Agentic Retrieval over the Entity Graph. There are no live results yet.")

### 4.3 Visualize the processing and timing from the activity log

The notebook displays the retrieved `entity_result["activity"]` in this order: **the full JSON, then a list by processing step, then a timing chart**. It does not search again or call the model for the visualization. Instead of narrowing the view to `graphSearch`, it also shows the records of search planning and reasoning.

| Log type | Processing to check |
|---|---|
| `modelQueryPlanning` | Search planning by the model, and the input and output token counts |
| `graphSearch` | Conversion from natural language to GQL, the target graph, the number of attempts, the outcome, and the model processing time |
| `searchIndex` / `azureBlob` | Knowledge source search and its search arguments |
| `agenticReasoning` | The reasoning setting at retrieval time and the reasoning token count |
| `modelAnswerSynthesis` | Answer generation. It is not necessarily present with `extractiveData` in this notebook |

If `startedAt` and `completedAt` exist, the notebook draws the timeline from those times so that you can also see processing that ran in parallel. For logs without start and end times, it shows a comparison chart of `elapsedMs` and does not guess the actual order or parallelism. It also does not substitute `queryTime` for a start time.

The time range of the chart is the range of the recorded processing, not the duration of the whole API response. The total of the processing times is not necessarily the API duration either. Token counts are shown as reported by each log, and values that are not recorded are not replaced with 0. `graphSearchArguments.modelElapsedMs` is shown in a separate column from the whole graph processing.

Compare the GQL that you wrote by hand in chapter 3 with the GQL generated this time. Even for the same question, the projected columns and the way the traversal is assembled do not necessarily match. The execution cell that follows stops when `graphSearch` is missing or reports a failure, so a response that contains only regular search is never treated as a successful graph search.

Source: [Activity array types and metrics](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#activity-array) / [Start and end times and overlapping operations](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#event-lifecycle) / [graphSearch response example](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
import math
from datetime import datetime


def display_activity(activities: list[dict], title: str, *, show_raw: bool = True) -> None:
    if not isinstance(activities, list):
        raise TypeError("activity must be an array. Check the SDK response.")
    print(title, "all activity entries:")
    if show_raw:
        show_json(activities)
    else:
        raw = html.escape(json.dumps(activities, ensure_ascii=False, indent=2))
        display(HTML(f'<details><summary>Open the raw activity JSON</summary><pre>{raw}</pre></details>'))
    if not activities:
        print("There are 0 activity entries, so the processing cannot be checked. Check include_activity.")
        return
    labels = {
        "modelQueryPlanning": "Search plan generation", "graphSearch": "GQL generation and graph search",
        "searchIndex": "Index search", "azureBlob": "Blob source search",
        "agenticReasoning": "Reasoning setting and usage", "modelAnswerSynthesis": "Answer generation",
        "modelWebSummarization": "Web search result summarization", "imageServing": "Image retrieval and delivery",
    }

    def timestamp(value: str | None) -> datetime | None:
        if value is None:
            return None
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        if parsed.tzinfo is None:
            raise ValueError("The activity timestamp has no time zone.")
        return parsed

    rows, records = [], []
    for activity in activities:
        kind = activity.get("type") or "Not recorded"
        graph = activity.get("graphSearchArguments") or {}
        model = activity.get("model") or {}
        start, end = timestamp(activity.get("startedAt")), timestamp(activity.get("completedAt"))
        if start is not None and end is not None and end < start:
            raise ValueError(f"activity {activity.get('id')}: the end time is earlier than the start time.")
        elapsed = activity.get("elapsedMs")
        basis = "elapsedMs"
        if elapsed is None and start is not None and end is not None:
            elapsed = (end - start).total_seconds() * 1000
            basis = "Computed from start and end times"
        if elapsed is not None and (isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0):
            raise ValueError(f"activity {activity.get('id')}: elapsedMs is invalid.")

        def metric(key: str) -> object:
            value = activity.get(key)
            return graph.get(key) if value is None else value

        row = {
            "ID": activity.get("id"), "Type": kind, "Processing": labels.get(kind, "Unregistered activity type"),
            "Source": activity.get("knowledgeSourceName"), "Model": model.get("modelName") or graph.get("modelName"),
            "Time ms": elapsed, "Time basis": basis if elapsed is not None else None,
            "Model time ms": graph.get("modelElapsedMs"),
            "Input tokens": metric("inputTokens"), "Output tokens": metric("outputTokens"),
            "Reasoning tokens": activity.get("reasoningTokens"), "Count": activity.get("count"),
            "Attempts": graph.get("attempts"), "Outcome or error": activity.get("error") or graph.get("outcome"),
        }
        rows.append({key: "Not recorded" if value is None else value for key, value in row.items()})
        records.append((activity, start, end, elapsed))
    table(rows)
    bounds = [(start, end) for _, start, end, _ in records if start is not None and end is not None]
    origin = min(start for start, _ in bounds) if bounds else None
    span = (max(end for _, end in bounds) - origin).total_seconds() * 1000 if origin is not None else max((elapsed for _, _, _, elapsed in records if elapsed is not None), default=0)
    caption = "Timeline: elapsed ms since the first start time" if origin is not None else "Time comparison: actual start times, order and parallelism are not recorded"
    print(caption)
    left, plot_width, height = 330, 550, 95 + 44 * len(records)
    scale = plot_width / max(span, 1)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1040 {height}" width="100%" role="img" aria-label="Processing time of {html.escape(title)}">',
             f'<text x="10" y="22" font-family="sans-serif" font-size="14">{html.escape(caption)}</text>']
    ticks = [span * index / 4 for index in range(5)] if span else [0]
    for tick in ticks:
        x = left + tick * scale
        parts.append(f'<line x1="{x}" y1="45" x2="{x}" y2="{height - 25}" stroke="#ddd"/>')
        parts.append(f'<text x="{x}" y="40" text-anchor="middle" font-family="sans-serif" font-size="12">{tick:.1f}</text>')
    for index, (activity, start, end, elapsed) in enumerate(records):
        y = 70 + 44 * index
        kind = activity.get("type") or "Not recorded"
        label = f"{activity.get('id', 'Not recorded')}: {labels.get(kind, kind)}"
        parts.append(f'<text x="10" y="{y + 5}" font-family="sans-serif" font-size="13">{html.escape(label)}</text>')
        if origin is not None and start is not None and end is not None:
            offset, duration = (start - origin).total_seconds() * 1000, (end - start).total_seconds() * 1000
        elif origin is None and elapsed is not None:
            offset, duration = 0, elapsed
        else:
            parts.append(f'<text x="{left}" y="{y + 5}" fill="#64748b" font-family="sans-serif" font-size="12">Start and end times not recorded</text>')
            continue
        x, width = left + offset * scale, duration * scale
        color = "#dc2626" if activity.get("error") or ((activity.get("graphSearchArguments") or {}).get("outcome") not in (None, "Success")) else "#3976b8"
        if duration == 0:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{color}"/>')
        else:
            parts.append(f'<rect x="{x}" y="{y - 9}" width="{width}" height="18" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{left + plot_width + 12}" y="{y + 5}" font-family="sans-serif" font-size="12">{duration:.1f} ms</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if RUN_AZURE:
    display_activity(entity_result["activity"], "Agentic Retrieval over the Entity Graph")
    graph_activities = [activity for activity in entity_result["activity"] if activity["type"] == "graphSearch"]
    if not graph_activities:
        show_json(entity_result["activity"])
        raise RuntimeError("Cannot confirm that graphSearch ran. Check the new KB, the source settings and the retrieval instructions.")
    for activity in graph_activities:
        arguments = activity["graphSearchArguments"]
        print("activity ID:", activity["id"], "Generated GQL:", arguments.get("gql", "Not retrieved"))
        if arguments.get("outcome") != "Success":
            raise RuntimeError("Cannot confirm that the graph processing succeeded. Check the execution history above.")
else:
    print("Not run: checking the graphSearch execution history and the generated GQL.")

### 4.4 Read the graph rows contained in the references

Graph rows may appear in `references[].sourceData.rows`. **Even when `response` is empty, the official example puts the results in `references`.** Do not decide that the search failed based only on the presence or absence of `response`.

The projected columns of the generated GQL depend on the model. The notebook therefore does not assume that the `designation`, `site` and `place` columns of the hand-written GQL exist, and displays the actual columns. Use `activitySource` to trace which search produced each reference.

Source: [Graph rows and references response example](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
if RUN_AZURE:
    retrieved_graph_rows = []
    activity_by_id = {activity["id"]: activity for activity in entity_result["activity"]}
    for reference in entity_result["references"]:
        source_data = reference.get("sourceData")
        if reference["type"] == "graphSearch" and isinstance(source_data, dict) and "rows" in source_data:
            source_activity = activity_by_id[reference["activitySource"]]
            table([{"Reference ID": reference["id"], "activity ID": source_activity["id"],
                    "Graph": reference["graphIndexName"], "Columns": source_data.get("columns", "Not retrieved")}])
            table(source_data["rows"])
            retrieved_graph_rows.extend(source_data["rows"])
    print("Number of graph rows read from the references:", len(retrieved_graph_rows))
    if not retrieved_graph_rows:
        print("No graph rows found. Check the execution history and the raw references data.")
        show_json(entity_result["references"])
else:
    print("Not run: checking the graph rows contained in the references.")

### 4.5 Read the evidence chunks and the URLs of the source documents

References that contain source text also have `type = graphSearch`, so you cannot tell them apart from graph rows by type alone. Pick out the references that have `sourceData.snippet`. `uid` is the chunk, `snippet_parent_id` is the parent document, and `blob_url` is the location of the source document.

The table shows the first 200 characters of the source text and truncates the rest with "…". The full text remains in `chunk_references` and `entity_result`.

The same `activitySource` shows that references come from the same search operation, but a source text reference does not necessarily prove a specific graph row one to one. To examine the edge mapping of each row in detail, use the evidence of the direct GQL in 3.4. A Blob URL is not necessarily a public URL, and viewing it may require permissions.

Source: [Graph references and source text retrieval through a knowledge base](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base)

In [ ]:
if RUN_AZURE:
    chunk_references = []
    for reference in entity_result["references"]:
        source_data = reference.get("sourceData")
        if reference["type"] == "graphSearch" and isinstance(source_data, dict) and "snippet" in source_data:
            chunk_references.append({"Reference ID": reference["id"], "activity ID": reference["activitySource"],
                                     "Chunk ID": source_data.get("uid", "Not retrieved"),
                                     "Parent document ID": source_data.get("snippet_parent_id", "Not retrieved"),
                                     "Source URL": source_data.get("blob_url", "Not retrieved"), "Source text": source_data["snippet"]})
    table([{**row, "Source text": preview_text(row["Source text"])} for row in chunk_references])
    if not chunk_references:
        print("No source text chunks were retrieved. Check the hydrate settings and the source fields.")
else:
    print("Not run: checking the evidence text returned by Agentic Retrieval.")

## 5. Reading the results and isolating problems

| Observed state | What to check next |
|---|---|
| There is no `graphSearch` | An Entity Graph needs a new KB. Check whether you connected the graph-enabled source when you created it, the retrieval instructions, and the actual activity |
| Direct GQL also returns 0 rows | Check that the sync is complete, the extracted types and predicates, the graph name and the original article. You cannot judge extraction success from an empty result |
| Direct GQL returns rows but natural-language retrieval is empty | Compare the generated GQL with your hand-written GQL, and check the differences in type names, predicates and name conditions |
| `502` and `All retrieval tasks failed` | The cause can be a `429` from the retrieval model. Check the model quota and the service errors, and do not resend without limit |
| `403` | Check separately the Search permissions of the running user, the model permissions of the Search managed identity, and the network conditions |
| `206` | This is a partial response. This notebook does not accept it as a success. Investigate from the displayed body and the request ID |
| `412` | A KB with the same name may have been created by another operation. Compare the plan in 4.1 with the definition of the existing KB, and specify a different name if needed. Do not change to an unconditional write |
| No source text | Check `includeReferenceSourceData`, `hydrate` and the source text field of the source. Also note the difference in hydration settings between direct GQL and the KB |

### Persisted changes and what to do afterward

The only change persisted in Azure is the dedicated knowledge base, and only if you allowed its first creation in 4.1. The notebook makes no change to the existing Blob source or the graph. The cell below only closes the clients, and the KB remains after the kernel ends.

Deletion is not automated. When you no longer need it, make sure that no app or agent references the KB created by this notebook, and delete only that KB. Do not delete resources that you reused.

The range of the graph and evidence chunks that you can retrieve does not necessarily match the viewing permissions of the source documents. Target only the sources that you are allowed to use, and check the access rights to both the source documents and the graph. The results of a notebook run can contain source text and connection targets, so clear the outputs before you share or commit the notebook.

Source: [Entity Graph troubleshooting](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#troubleshoot-graph-queries) / [Retrieval permissions](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [Concurrency control](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE:
    for client in azure_clients:
        client.close()
    credential.close()
    print("Closed the SDK and credential clients. No Azure resources were deleted.")
else:
    print("No Azure credential client was created.")